# Product Review Intelligence — Demo Notebook

**S8 Integrated Project · UIR · 2025–2026**  
**Authors:** Chaymae Benmakhlouf · Maroua Ait Sassi · Rania Bouaroua  
**Supervised by:** Prof. Hakim Hafidi  

---

A multi-agent AI system that:
1. Classifies product reviews with a **fine-tuned DistilBERT** (94.5% accuracy on the held-out test set).
2. Scouts competitors via **DuckDuckGo web search**.
3. Synthesises a one-page **market brief**, with a **human-in-the-loop** checkpoint.

**This notebook runs the full pipeline end-to-end.**  
Estimated runtime on Colab GPU (T4): **≈ 5 minutes**.

Before running: in the Colab menu, go to **Runtime → Change runtime type → T4 GPU**.


## 1.  Setup — clone the repo and install dependencies


In [ ]:
# Clone the project repository

!git clone https://github.com/aitsassimaroua-dot/uir-product-review-intelligence.git

%cd uir-product-review-intelligence



# Install dependencies (silent)

!pip install -q -r requirements.txt



# Generate the 10-review sample CSV used by the demo

!python scripts/make_sample_reviews.py

print('Setup complete.')


## 2.  Fine-tune DistilBERT on `amazon_polarity`

We fine-tune `distilbert-base-uncased` on a stratified subset of `amazon_polarity`. For the demo we use a **1-epoch quick run** on 5,000 training examples — completes in roughly 2 minutes on a T4 GPU.

The full configuration in the report (18k train × 2 epochs) reaches **94.5% accuracy** and **0.945 weighted F1** on 5,000 held-out test examples.

In [ ]:
# Quick training run — 1 epoch, 5,000 examples

!python -m src.model.train --epochs 1 --subset 5000


## 3.  Evaluate the model

Compute accuracy, F1, precision/recall per class, and the confusion matrix on the held-out test set.

In [ ]:
import json

from pathlib import Path

from IPython.display import Image, display



# Evaluate on the held-out test set

!python -m src.model.evaluate



metrics = json.loads(Path('outputs/eval_metrics.json').read_text())

print('\nTest metrics')

print('-' * 30)

for k, v in metrics.items():

    if isinstance(v, float):

        print(f'  {k:20s}  {v:.4f}')

    elif k != 'confusion_matrix':

        print(f'  {k:20s}  {v}')



# Show the classification report

print()

print(Path('outputs/eval_classification_report.txt').read_text())



# Show the confusion matrix

display(Image('outputs/eval_confusion_matrix.png'))


## 4.  Configure the LLM (Gemini)

The agents need an LLM. We use Google's **Gemini 1.5 Flash** — free, no credit card required.

1. Go to https://aistudio.google.com/apikey
2. Create a key (takes 30 seconds)
3. Paste it in the cell below, then run it

In [ ]:
import os



# ┌─────────────────────────────────────────────────────────────────┐

# │  Paste your free Gemini API key here:                           │

# │  (get one at https://aistudio.google.com/apikey)                │

# └─────────────────────────────────────────────────────────────────┘

os.environ['GEMINI_API_KEY'] = ''   # ← paste your key between the quotes

os.environ['MODEL'] = 'gemini/gemini-1.5-flash'



assert os.environ['GEMINI_API_KEY'], (

    'Paste a Gemini API key above and re-run this cell.'

)

print('LLM configured.')


## 5.  Run the multi-agent pipeline

Three agents collaborate:

1. **Sentiment Analyst** — owns the trained DistilBERT, classifies every review, extracts top complaints and praises.
2. **Market Researcher** — queries DuckDuckGo using the analyst's complaints as seeds, returns three plausible competitors with real evidence URLs.
3. **Report Orchestrator** — synthesises both outputs into a one-page market brief.

We disable the interactive human-in-the-loop here so the notebook runs unattended.

In [ ]:
from src.crew import build_crew

from src.utils.logging_config import configure_logging



configure_logging(run_id='colab_demo')



crew = build_crew(

    product_name='Wireless Earbuds X',

    reviews_path='data/processed/sample_reviews.csv',

    with_hitl=False,   # skip the interactive approval prompt

)

result = crew.kickoff()

print('\nPipeline finished.')


## 6.  The generated market brief

The orchestrator's final brief is saved to `outputs/market_brief_<timestamp>.md`. Below we render the latest one inline.

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display



briefs = sorted(Path('outputs').glob('market_brief_*.md'))

if not briefs:

    print('No brief found — re-run the previous cell.')

else:

    latest = briefs[-1]

    print(f'Brief file: {latest.name}\n')

    display(Markdown(latest.read_text()))


## 7.  Inspect the JSONL action log

Every tool call is appended to `logs/agent_actions_<run_id>.jsonl` with input hash, output label, latency, and timestamp. This is the audit trail.

In [ ]:
import json

from pathlib import Path



logs = sorted(Path('logs').glob('agent_actions_*.jsonl'))

if logs:

    lines = logs[-1].read_text().strip().split('\n')

    print(f'Log file: {logs[-1].name}  ({len(lines)} entries)\n')

    # Show the first 5 tool.call entries

    shown = 0

    for line in lines:

        try:

            rec = json.loads(line)

        except Exception:

            continue

        if rec.get('msg') == 'tool.call':

            print(json.dumps(rec, indent=2))

            print()

            shown += 1

            if shown >= 5:

                break

else:

    print('No log file found.')


---

## End of demo

**What this notebook demonstrated**

- Fine-tuned a DistilBERT classifier on `amazon_polarity` and evaluated it.
- Ran the 3-agent CrewAI pipeline on a 10-review CSV.
- Produced a structured market brief with the trained model used as a CrewAI tool.
- Logged every agent action as JSONL for auditability.

**The full report, slides, and demo video** are also in the repository.

*Thank you for reviewing our work.*